# SHAP Explainability
## Telco Customer Churn Prediction

This is probably the most important notebook in the project from a business perspective.
SHAP answers the question that feature importance can't: not just *which* features matter,
but *how* and *why* — for every single prediction.

Sections:
1. What is SHAP and why does it matter?
2. Global feature importance — beeswarm and bar plots
3. Dependence plots — how each top feature affects predictions
4. Individual prediction explanations — four real examples
5. Business recommendations from SHAP insights
6. High-risk customer profile summary

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

import shap

from src.preprocessing import load_data, clean_data, split_data
from src.features import engineer_features, get_feature_names, create_preprocessor
from src.explain import (
    compute_shap_values, plot_beeswarm, plot_bar_summary,
    plot_dependence, plot_waterfall, get_top_churn_drivers,
    find_example_indices,
)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})
shap.initjs()

## 1. What Is SHAP and Why Does It Matter?

Regular feature importance (from tree models) tells you which features the model used the most overall.
SHAP goes further — it uses a concept from cooperative game theory called Shapley values to assign each
feature a contribution to each individual prediction.

The difference matters in practice:

| | Feature Importance | SHAP |
|---|---|---|
| Scope | Global only | Global AND per-prediction |
| Direction | No (just magnitude) | Yes (pushes toward or away from churn) |
| Interactions | Hidden | Visible via dependence plots |
| Explainability | "Contract type is important" | "This customer's month-to-month contract increased their churn probability by 0.28" |

For a business, SHAP means you can explain to a manager exactly why a specific customer was flagged.

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = engineer_features(clean_data(load_data(RAW_PATH)))
X_train, X_test, y_train, y_test = split_data(df)

preprocessor = joblib.load("../models/preprocessor.joblib")
X_train_proc  = preprocessor.transform(X_train)
X_test_proc   = preprocessor.transform(X_test)

best_model = joblib.load("../models/best_model.joblib")
with open("../models/model_meta.json") as f:
    meta = json.load(f)

print(f"Model: {meta['model_name']}")
print(f"Test F1: {meta['tuned_f1_test']}  |  ROC-AUC: {meta['tuned_roc_auc_test']}")

# get feature names after one-hot encoding
numerical_cols = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "monthly_tenure_ratio", "total_services", "avg_monthly_spend", "contract_risk",
]
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod", "tenure_group",
]
try:
    feature_names = get_feature_names(preprocessor, numerical_cols, categorical_cols)
except Exception:
    feature_names = [f"f{i}" for i in range(X_test_proc.shape[1])]

print(f"\nFeature matrix: {X_test_proc.shape[1]} features")

In [ ]:
# compute SHAP values on a sample of the test set (500 is enough for global plots)
print("Computing SHAP values (this takes ~30 seconds)...")
explainer, shap_values, X_sample = compute_shap_values(
    best_model, X_test_proc, sample_size=500
)
print(f"Done. SHAP values computed for {X_sample.shape[0]} samples.")
print(f"Expected value (base rate): {explainer.expected_value:.4f}")

## 2. Global Feature Importance

### Beeswarm Plot

This is the most informative chart in the whole project.

Each row is one feature. Each dot is one prediction.
- **X position:** how much that feature pushed the prediction up (toward churn) or down (away from churn)
- **Colour:** the actual value of that feature for that customer (red = high, blue = low)

So you can read it as: "customers with low tenure (blue dots) are pushed to the right — toward churn."
That's a much richer story than "tenure has importance 0.15".

In [ ]:
plot_beeswarm(shap_values, max_display=20, save_path="../reports/fig_20_shap_beeswarm.png")

Reading the beeswarm:

- **Contract type (Month-to-month):** red dots on the far right — customers with month-to-month
  contracts get pushed strongly toward churn. The single biggest driver.
- **tenure:** blue dots on the right — low tenure pushes toward churn. Long-tenure customers
  (red) are pushed away from churn. Makes sense: loyal customers are less likely to leave.
- **InternetService (Fiber optic):** red dots on the right — fiber optic customers churn more.
  Could be pricing or service quality expectations not being met.
- **MonthlyCharges:** higher charges (red) push toward churn — customers paying more are
  more likely to shop around.
- **TechSupport (No):** customers without tech support are pushed toward churn, suggesting
  these customers feel underserved.

In [ ]:
# bar plot — simpler summary version
plot_bar_summary(shap_values, max_display=15, save_path="../reports/fig_21_shap_bar.png")

In [ ]:
# top churn drivers as a table
top_drivers = get_top_churn_drivers(shap_values, feature_names, top_n=10)
print("Top 10 churn drivers by mean absolute SHAP value:")
display(top_drivers)

## 3. Dependence Plots

Dependence plots show how a single feature's value relates to its SHAP contribution.
The colour shows a second feature, which reveals interactions.

In [ ]:
# tenure — coloured by contract type (if available as a feature)
# find the index of tenure in feature_names
tenure_idx = feature_names.index("tenure") if "tenure" in feature_names else 0

fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(
    tenure_idx,
    shap_values.values,
    X_sample,
    feature_names=feature_names,
    ax=ax,
    show=False,
)
ax.set_title("SHAP Dependence — tenure", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("SHAP value (impact on churn prediction)")
plt.tight_layout()
plt.savefig("../reports/fig_22_shap_dep_tenure.png", bbox_inches="tight")
plt.show()

The curve drops sharply in the first 12 months — new customers are at the highest risk.
After about 24 months the SHAP value stabilises near zero or negative, meaning tenure stops
being a positive churn driver. Long-tenure customers are actively predicted *away* from churn.

In [ ]:
# MonthlyCharges dependence
mc_idx = feature_names.index("MonthlyCharges") if "MonthlyCharges" in feature_names else 1

fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(
    mc_idx,
    shap_values.values,
    X_sample,
    feature_names=feature_names,
    ax=ax,
    show=False,
)
ax.set_title("SHAP Dependence — MonthlyCharges", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Monthly Charges ($)")
ax.set_ylabel("SHAP value (impact on churn prediction)")
plt.tight_layout()
plt.savefig("../reports/fig_23_shap_dep_monthly_charges.png", bbox_inches="tight")
plt.show()

Monthly charges above roughly $65-70 start consistently pushing predictions toward churn.
Below that range the effect is mostly neutral or slightly negative.

In [ ]:
# contract_risk engineered feature
cr_idx = feature_names.index("contract_risk") if "contract_risk" in feature_names else 2

fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(
    cr_idx,
    shap_values.values,
    X_sample,
    feature_names=feature_names,
    ax=ax,
    show=False,
)
ax.set_title("SHAP Dependence — contract_risk (0=Two year, 1=One year, 2=Month-to-month)",
             fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Contract Risk Score")
ax.set_ylabel("SHAP value (impact on churn prediction)")
plt.tight_layout()
plt.savefig("../reports/fig_24_shap_dep_contract_risk.png", bbox_inches="tight")
plt.show()

## 4. Individual Prediction Explanations

Waterfall plots explain *one specific prediction*. They show:
- The base value (average model output across all data)
- Each feature either pushing up (red) or down (blue) toward the final prediction
- The final predicted churn probability

We look at four different types of predictions.

In [ ]:
# find representative examples
indices = find_example_indices(best_model, X_test_proc, y_test,
                               optimal_threshold=meta["optimal_threshold"])

y_prob = best_model.predict_proba(X_test_proc)[:, 1]
X_test_reset = X_test.reset_index(drop=True)

for label, idx in indices.items():
    prob   = y_prob[idx]
    actual = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]
    tenure = X_test_reset.iloc[idx]["tenure"]
    contract = X_test_reset.iloc[idx]["Contract"]
    monthly  = X_test_reset.iloc[idx]["MonthlyCharges"]
    print(f"{label:15s} | idx={idx:4d} | prob={prob:.3f} | actual={'Churn' if actual else 'Stay':5s}"
          f" | tenure={tenure:3.0f}mo | contract={contract:20s} | monthly=${monthly:.0f}")

In [ ]:
# we need shap_values indexed on the full test set, not just the 500-sample subset
# recompute on full test set for the specific indices we want
full_explainer = shap.TreeExplainer(best_model)
all_indices    = list(indices.values())
X_examples     = X_test_proc[all_indices]
sv_examples    = full_explainer(X_examples)

In [ ]:
# 1. High-risk churner we caught correctly
example_pos = 0
actual_label = "Churn" if (y_test.iloc[indices["high_risk_tp"]]
                           if hasattr(y_test, 'iloc') else y_test[indices["high_risk_tp"]]) else "Stay"

print(f"Example 1 — High-risk churner (caught correctly)")
print(f"Predicted probability: {y_prob[indices['high_risk_tp']]:.3f}  |  Actual: {actual_label}")
print(f"Tenure: {X_test_reset.iloc[indices['high_risk_tp']]['tenure']:.0f} months  "
      f"Contract: {X_test_reset.iloc[indices['high_risk_tp']]['Contract']}  "
      f"Monthly: ${X_test_reset.iloc[indices['high_risk_tp']]['MonthlyCharges']:.0f}")

plt.figure(figsize=(11, 5))
shap.plots.waterfall(sv_examples[example_pos], max_display=12, show=False)
plt.title("Example 1 — High-risk churner (correctly caught)", fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../reports/fig_25_shap_waterfall_tp.png", bbox_inches="tight")
plt.show()

In [ ]:
# 2. Low-risk customer who stayed
example_pos = 1
actual_label = "Churn" if (y_test.iloc[indices["low_risk_tn"]]
                           if hasattr(y_test, 'iloc') else y_test[indices["low_risk_tn"]]) else "Stay"

print(f"Example 2 — Low-risk customer (correctly predicted to stay)")
print(f"Predicted probability: {y_prob[indices['low_risk_tn']]:.3f}  |  Actual: {actual_label}")
print(f"Tenure: {X_test_reset.iloc[indices['low_risk_tn']]['tenure']:.0f} months  "
      f"Contract: {X_test_reset.iloc[indices['low_risk_tn']]['Contract']}  "
      f"Monthly: ${X_test_reset.iloc[indices['low_risk_tn']]['MonthlyCharges']:.0f}")

plt.figure(figsize=(11, 5))
shap.plots.waterfall(sv_examples[example_pos], max_display=12, show=False)
plt.title("Example 2 — Low-risk customer (correctly predicted to stay)", fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../reports/fig_26_shap_waterfall_tn.png", bbox_inches="tight")
plt.show()

In [ ]:
# 3. False negative — churner we missed
example_pos = 2
actual_label = "Churn" if (y_test.iloc[indices["false_neg"]]
                           if hasattr(y_test, 'iloc') else y_test[indices["false_neg"]]) else "Stay"

print(f"Example 3 — False negative (churner we missed)")
print(f"Predicted probability: {y_prob[indices['false_neg']]:.3f}  |  Actual: {actual_label}")
print(f"Tenure: {X_test_reset.iloc[indices['false_neg']]['tenure']:.0f} months  "
      f"Contract: {X_test_reset.iloc[indices['false_neg']]['Contract']}  "
      f"Monthly: ${X_test_reset.iloc[indices['false_neg']]['MonthlyCharges']:.0f}")
print("Why did we miss this churner? Look at which features pushed the prediction down.")

plt.figure(figsize=(11, 5))
shap.plots.waterfall(sv_examples[example_pos], max_display=12, show=False)
plt.title("Example 3 — False negative (missed churner)", fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../reports/fig_27_shap_waterfall_fn.png", bbox_inches="tight")
plt.show()

In [ ]:
# 4. False positive — non-churner we flagged
example_pos = 3
actual_label = "Churn" if (y_test.iloc[indices["false_pos"]]
                           if hasattr(y_test, 'iloc') else y_test[indices["false_pos"]]) else "Stay"

print(f"Example 4 — False positive (non-churner we flagged)")
print(f"Predicted probability: {y_prob[indices['false_pos']]:.3f}  |  Actual: {actual_label}")
print(f"Tenure: {X_test_reset.iloc[indices['false_pos']]['tenure']:.0f} months  "
      f"Contract: {X_test_reset.iloc[indices['false_pos']]['Contract']}  "
      f"Monthly: ${X_test_reset.iloc[indices['false_pos']]['MonthlyCharges']:.0f}")
print("Why did we flag this customer? Which features pushed the prediction up even though they stayed?")

plt.figure(figsize=(11, 5))
shap.plots.waterfall(sv_examples[example_pos], max_display=12, show=False)
plt.title("Example 4 — False positive (non-churner we flagged)", fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../reports/fig_28_shap_waterfall_fp.png", bbox_inches="tight")
plt.show()

## 5. Business Recommendations

These recommendations come directly from the SHAP analysis — not guesswork.

**1. Contract upgrade campaign (highest priority)**
Month-to-month contracts are the single largest churn driver. Offering discounts or
loyalty incentives to move customers onto annual or two-year contracts would reduce
the biggest SHAP driver across the board.

**2. Early-tenure intervention**
The first 12 months are the highest-risk window. A structured onboarding programme —
check-in calls, tutorials, satisfaction surveys — for customers in their first year
would target the period where tenure's SHAP contribution is most negative (i.e. riskiest).

**3. Investigate fiber optic service quality**
Fiber optic customers churn more than DSL customers, and SHAP confirms this is a
meaningful signal rather than just correlation with price. A service quality review
or customer satisfaction survey focused on fiber optic customers would identify
whether this is a speed, reliability, or expectation issue.

**4. Bundle security and support services**
Customers without OnlineSecurity, TechSupport, or OnlineBackup have higher SHAP
contributions toward churn. A discounted bundle offer targeted at customers missing
these services could reduce churn and increase average revenue per user simultaneously.

**5. High-value, month-to-month customers are the top priority segment**
Monthly charges above ~$70 combined with month-to-month contracts and tenure under
12 months defines the highest-risk profile in the SHAP analysis. These customers
should be the first cohort for any retention campaign.

In [ ]:
# high-risk profile summary chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# top drivers bar chart
top10 = get_top_churn_drivers(shap_values, feature_names, top_n=10)
colors = sns.color_palette("RdYlGn_r", len(top10))
axes[0].barh(top10["feature"][::-1], top10["mean_abs_shap"][::-1],
             color=colors, edgecolor="white")
axes[0].set_xlabel("Mean |SHAP value|")
axes[0].set_title("Top 10 Churn Drivers (SHAP)", fontweight="bold")
for i, (_, row) in enumerate(top10.iloc[::-1].iterrows()):
    axes[0].text(row["mean_abs_shap"] + 0.001, i,
                 f"{row['mean_abs_shap']:.4f}", va="center", fontsize=8)

# high-risk profile definition
risk_profile = {
    "Contract":         "Month-to-month",
    "Tenure":           "< 12 months",
    "MonthlyCharges":   "> $70",
    "InternetService":  "Fiber optic",
    "TechSupport":      "No",
    "OnlineSecurity":   "No",
}
y_pos  = range(len(risk_profile))
labels = list(risk_profile.keys())
values = list(risk_profile.values())

axes[1].barh(y_pos, [1] * len(risk_profile),
             color=sns.color_palette("Reds", len(risk_profile))[::-1],
             edgecolor="white", height=0.6)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(labels, fontsize=11)
axes[1].set_xticks([])
for i, v in enumerate(values):
    axes[1].text(0.5, i, v, ha="center", va="center",
                 fontsize=10, fontweight="bold", color="white")
axes[1].set_title("Highest-Risk Customer Profile
(defined by SHAP analysis)",
                  fontweight="bold")
axes[1].set_xlim(0, 1)

plt.suptitle("SHAP Summary — Churn Drivers and Risk Profile", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_29_shap_summary_card.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# estimate retention opportunity
y_prob_all    = best_model.predict_proba(X_test_proc)[:, 1]
threshold     = meta["optimal_threshold"]
n_test        = len(y_test)
n_total       = len(df)
scale_factor  = n_total / n_test

high_risk_mask = y_prob_all >= 0.70        # top-risk customers
n_high_risk    = high_risk_mask.sum()
n_high_risk_scaled = int(n_high_risk * scale_factor)

avg_monthly_charge = df["MonthlyCharges"].mean()
avg_annual_revenue = avg_monthly_charge * 12

print("Retention opportunity estimate (rough):")
print(f"  Total customers in dataset:        {n_total:,}")
print(f"  High-risk customers (prob >= 0.70): {n_high_risk_scaled:,} ({n_high_risk_scaled/n_total:.1%})")
print(f"  Average annual revenue per customer: ${avg_annual_revenue:,.0f}")
print(f"  Potential annual revenue at risk:  "
      f"${n_high_risk_scaled * avg_annual_revenue:,.0f}")
print()
print("If a retention campaign converts even 20% of high-risk customers:")
print(f"  Customers retained: ~{int(n_high_risk_scaled * 0.20):,}")
print(f"  Revenue protected:  ~${int(n_high_risk_scaled * 0.20 * avg_annual_revenue):,}")